In [81]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import polars as pl
plt.style.use('fivethirtyeight')
sns.set_context('notebook')


In [3]:
with open('names.txt') as file:
    names = [line.rstrip() for line in file]
print(len(names))
print(max(len(w) for w in names))
print(names[:8])
    

32033
15
['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']


What does a bigram model do? It learns to predict the next character from the preceding character.

So what are the steps we need to take?
- Divide the dataset into individual names (done)
- Load up individual two char pairs: This includes a padding character (say ".")
- Set up our dictionary containing all characters in the dataset + the padding character
- For every character, count the number of times each continuation happens in the dataset ("Training"). Add + 1 to all bigrams to avoid zero-probabilities in our model 

That's it. That's the model. What's missing still is evaluation:
- Generate continuations - either most likely token or sampled by probability
- Calculate **loss**: We can use the usual loss function, which is...well we have categorical data, so our loss function is cross-entropy loss, given by  -log P(x), that is the negative log likelihood it assigned to the correct token

In [ ]:
chars = set()
for name in names:
    for char in name:
        chars.add(char)
chars.add(".")
chars = list(sorted(chars))
# encoding = {char : number for number, char in enumerate(chars)}
# decoding = {number : char for number, char in enumerate(chars)}

In [ ]:
counts = {char : {char : 1 for char in chars} for char in chars} # initialize dict with all values at 1 to avoid infinite loss

# "Training loop":
for name in names:
    name = "." + name + "."
    for c1, c2 in zip(name, name[1:]):
        counts[c1][c2] += 1

In [80]:
# Calculate loss over whole dataset:
loss = 0
bigram_count = 0
for name in names:
    name = "." + name + "."
    for c1, c2 in zip(name, name[1:]):
        prob = counts[c1][c2] / sum(counts[c1].values())
        loss += -np.log(prob)
        bigram_count += 1
loss = loss / bigram_count
print(loss)

2.454576820116076


In [77]:
# Generate a name:
mode = 'sample'
name = "."
while True:
    if mode == 'max':
        name += max(chars, key = lambda c : counts[name[-1]][c])
    elif mode == 'sample':
        name += np.random.choice(chars, p = np.array(list(counts[name[-1]].values())) / sum(counts[name[-1]].values()))
    if name[-1] == ".":
        break
name[1:-1]

'ka'

In [101]:
count_df = pl.DataFrame([(c1, c2, count) for c1, row in counts.items() for c2, count in row.items()], schema = ['c1', 'c2', 'count'], orient='row')
count_df.head()
                                  

c1,c2,count
str,str,i64
""".""",""".""",1
""".""","""a""",4411
""".""","""b""",1307
""".""","""c""",1543
""".""","""d""",1691


In [99]:
pl.Config.set_tbl_rows(30)
count_df.pivot(index='c1', on='c2', values='count')

c1,.,a,b,c,d,e,f,g,h,i,j,k,l,m,n,o,p,q,r,s,t,u,v,w,x,y,z
str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
""".""",1,4411,1307,1543,1691,1532,418,670,875,592,2423,2964,1573,2539,1147,395,516,93,1640,2056,1309,79,377,308,135,536,930
"""a""",6641,557,542,471,1043,693,135,169,2333,1651,176,569,2529,1635,5439,64,83,61,3265,1119,688,382,835,162,183,2051,436
"""b""",115,322,39,2,66,656,1,1,42,218,2,1,104,1,5,106,1,1,843,9,3,46,1,1,1,84,1
"""c""",98,816,1,43,2,552,1,3,665,272,4,317,117,1,1,381,2,12,77,6,36,36,1,1,4,105,5
"""d""",517,1304,2,4,150,1284,6,26,119,675,10,4,61,31,32,379,1,2,425,30,5,93,18,24,1,318,2
"""e""",3984,680,122,154,385,1272,83,126,153,819,56,179,3249,770,2676,270,84,15,1959,862,581,70,464,51,133,1071,182
"""f""",81,243,1,1,1,124,45,2,2,161,1,3,21,1,5,61,1,1,115,7,19,11,1,5,1,15,3
"""g""",109,331,4,1,20,335,2,26,361,191,4,1,33,7,28,84,1,1,202,31,32,86,2,27,1,32,2
"""h""",2410,2245,9,3,25,675,3,3,2,730,10,30,186,118,139,288,2,2,205,32,72,167,40,11,1,214,21
